# `receiving_tds` — modelos de conteo, no una regresión genérica

## Por qué este target necesita su propio notebook

`04_eda_targets.ipynb` (Fase 3) ya mostró que `receiving_tds` es fundamentalmente distinto:
**82% de las semanas en cero, incluso en receptores activos**. El pipeline heredado lo trató igual
que yardas/recepciones — una regresión genérica — y el resultado real fue un **R² de -0.087 en
prueba** (peor que adivinar el promedio, ver `docs/PLAN.md`). Este notebook responde dos preguntas:
¿hay una técnica mejor que una regresión genérica para un conteo con exceso de ceros, y de haberla,
cuál es?, y ¿conviene entrenar con todas las semanas o solo con las activas, para este target en
particular?

In [1]:
import sys
sys.path.insert(0, "../src")
import data, features, modeling, experimentos

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor, TweedieRegressor
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_poisson_deviance
from xgboost import XGBRegressor

pd.set_option("display.width", 160)
sns.set_style("whitegrid")
AZUL, NARANJA, VERDE, GRIS, GRIS_CLARO = "#2a78d6", "#eb6834", "#1baf7a", "#888780", "#c3c2b7"

wr = features.construir_tabla_modelado(range(2016, 2026))
features_arbol = features.columnas_por_modelo("arbol")
features_ridge = features.columnas_por_modelo("ridge")
train_mask, val_mask, test_mask = experimentos.dividir_temporal(
    wr, range(2016, 2022), [2022, 2023], [2024, 2025]
)
y = wr["receiving_tds"]
print(f"receiving_tds: min={y.min()}, max={y.max()} -- {(y == 0).mean():.1%} de las filas en cero")

receiving_tds: min=0, max=4 -- 81.8% de las filas en cero


## 1. Los modelos genéricos, primero (mismo punto de partida que recepciones/yardas)

In [2]:
resultados_genericos, resumen_genericos = experimentos.entrenar_y_evaluar(
    wr, "receiving_tds", features_arbol, features_ridge, train_mask, val_mask, test_mask
)
resumen_genericos.round(4)

,train_mae,train_rmse,train_r2,val_mae,val_rmse,val_r2,test_mae,test_rmse,test_r2
Baseline,0.3193,0.5058,-0.1105,0.2935,0.4795,-0.1192,0.2940,0.4834,-0.0711
RandomForest,0.3111,0.4444,0.1428,0.3093,0.4353,0.0779,0.3113,0.4473,0.0827
HistGradientBoosting,0.3066,0.4361,0.1747,0.3103,0.4370,0.0705,0.3099,0.4499,0.0720
XGBoost,0.2924,0.4198,0.2351,0.3104,0.4381,0.0658,0.3071,0.4522,0.0625
Ridge,0.3234,0.4609,0.0780,0.3109,0.4366,0.0720,0.3088,0.4469,0.0845


Los 5 mejoran al legado (que tuvo R² negativo en prueba) — pero eso es una vara baja. La pregunta
real es si una técnica pensada para conteos con exceso de ceros hace mejor trabajo que una
regresión genérica optimizada para error cuadrático.

## 2. Modelos de conteo — Poisson, Tweedie, y sus versiones en árbol

- **`PoissonRegressor`**: supone que la varianza del conteo es igual a su media — el modelo de
  conteo más simple.
- **`TweedieRegressor(power=1.5)`**: más flexible que Poisson si hay sobre-dispersión (varianza
  mayor que la media).
- **`XGBRegressor(objective="count:poisson")`** y **`XGBRegressor(objective="reg:tweedie")`**: los
  mismos supuestos de conteo, pero dentro de un árbol de gradient boosting, que además puede
  capturar interacciones que un GLM lineal no ve.
- **`HistGradientBoostingRegressor(loss="poisson")`**: mismo modelo ya usado en Fase 4, con la
  función de pérdida correcta para este problema en vez de la genérica.

In [3]:
modelos_conteo = {
    "Poisson": Pipeline([("imputar", SimpleImputer(strategy="median")), ("modelo", PoissonRegressor(max_iter=500))]),
    "Tweedie (power=1.5)": Pipeline([("imputar", SimpleImputer(strategy="median")), ("modelo", TweedieRegressor(power=1.5, max_iter=500))]),
    "XGBoost (count:poisson)": XGBRegressor(objective="count:poisson", random_state=0, n_jobs=-1),
    "XGBoost (reg:tweedie)": XGBRegressor(objective="reg:tweedie", tweedie_variance_power=1.5, random_state=0, n_jobs=-1),
    "HGB (loss=poisson)": HistGradientBoostingRegressor(loss="poisson", random_state=0),
}
X_arbol = wr[features_arbol]
resultados_conteo = {
    nombre: experimentos.evaluar_modelo(modelo, X_arbol, y, train_mask, val_mask, test_mask)
    for nombre, modelo in modelos_conteo.items()
}

/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_glm/glm.py:268: OptimizeWarning: Unknown solver options: iprint
  opt_res = scipy.optimize.minimize(


/usr/local/lib/python3.13/site-packages/sklearn/linear_model/_glm/glm.py:268: OptimizeWarning: Unknown solver options: iprint
  opt_res = scipy.optimize.minimize(


## 3. Modelo de 2 etapas (hurdle) — la técnica pensada específicamente para 82% de ceros

Etapa 1: clasifica P(anota al menos un TD | features). Etapa 2: regresa el conteo esperado
condicional a haber anotado, entrenado solo sobre filas con `receiving_tds > 0`. Predicción final
= P(etapa 1) × E[etapa 2] — no fuerza a un solo modelo a explicar "casi siempre cero" y "a veces
2-3" con la misma función.

In [4]:
resultados_conteo["Hurdle"] = experimentos.entrenar_hurdle(
    wr, "receiving_tds", features_arbol, train_mask, val_mask, test_mask
)

## 4. Tabla unificada — MAE (principal) y deviance de Poisson (secundaria)

MAE es la métrica principal, consistente con el resto de Fase 5 — pero para un conteo con exceso
de ceros vale la pena ver también la *deviance* de Poisson, que castiga distinto el error relativo
cerca de cero. Se calculan las dos, sobre el mismo conjunto de prueba, para todos los modelos
(genéricos + de conteo + hurdle).

In [5]:
todos_los_resultados = {**resultados_genericos, **resultados_conteo}
tabla_final = []
for nombre, salida in todos_los_resultados.items():
    pred_test_clip = np.clip(salida["pred"][test_mask], 1e-6, None)
    tabla_final.append({
        "modelo": nombre,
        "test_mae": salida["test"]["mae"],
        "test_r2": salida["test"]["r2"],
        "test_deviance_poisson": mean_poisson_deviance(y[test_mask], pred_test_clip),
    })
tabla_final = pd.DataFrame(tabla_final).set_index("modelo").sort_values("test_mae")
tabla_final.round(4)

,test_mae,test_r2,test_deviance_poisson
modelo,,,
XGBoost (reg:tweedie),0.2466,-0.0454,0.9467
XGBoost (count:poisson),0.2925,0.0536,0.6542
Baseline,0.2940,-0.0711,1.7828
Hurdle,0.3063,0.0690,0.6391
XGBoost,0.3071,0.0625,0.6392
HGB (loss=poisson),0.3075,0.0756,0.6332
Tweedie (power=1.5),0.3079,0.0781,0.6367
Ridge,0.3088,0.0845,0.6340
HistGradientBoosting,0.3099,0.0720,0.6389


## 5. Lectura honesta — MAE y deviance no siempre están de acuerdo

In [6]:
ganador_mae = tabla_final["test_mae"].idxmin()
ganador_deviance = tabla_final["test_deviance_poisson"].idxmin()
print(f"Mejor por MAE: {ganador_mae} ({tabla_final.loc[ganador_mae, 'test_mae']:.4f})")
print(f"Mejor por deviance de Poisson: {ganador_deviance} ({tabla_final.loc[ganador_deviance, 'test_deviance_poisson']:.4f})")
print(f"Deviance de {ganador_mae} (el mejor por MAE): {tabla_final.loc[ganador_mae, 'test_deviance_poisson']:.4f}")

Mejor por MAE: XGBoost (reg:tweedie) (0.2466)
Mejor por deviance de Poisson: RandomForest (0.6310)
Deviance de XGBoost (reg:tweedie) (el mejor por MAE): 0.9467


Los dos criterios **no coinciden** — es un hallazgo real, no un error de cálculo. MAE y deviance de
Poisson castigan distinto: MAE trata por igual un error de 0.3 en cualquier predicción; la
deviance castiga más fuerte un error relativo grande cerca de cero (predecir 0.1 cuando el real es
1 pesa más que predecir 2.1 cuando el real es 3, aunque el error absoluto sea el mismo).

Hay una tercera señal que tampoco coincide con el MAE: el **R² de `XGBoost (reg:tweedie)` es
-0.0454** — prácticamente empatado con el del Baseline (-0.0711), y peor que Ridge (0.0845) o
RandomForest (0.0827), que sí tienen mejor R² pese a peor MAE. Un modelo que gana en MAE pero
apenas supera al baseline en R² no es un ganador incuestionable — es una señal de que "MAE más
bajo" y "explica mejor la variación real del target" no son la misma pregunta para este target en
particular. Con MAE como métrica principal —la misma que se usó para elegir en
`receptions`/`receiving_yards`, para no cambiar de criterio a mitad de la fase— se recomienda
`XGBoost (reg:tweedie)`, pero con esta salvedad documentada explícitamente, no oculta: si más
adelante `receiving_tds` necesita explicar varianza y no solo minimizar error absoluto, vale la
pena reconsiderar entre `XGBoost (reg:tweedie)` y una alternativa con mejor R² (`Ridge` o
`RandomForest`).

## 6. ¿Todas las semanas o solo las activas? — la decisión pospuesta de Fase 4, para este target

Misma metodología correcta que en `13_comparacion_modelos.ipynb`: mismo conjunto de prueba (solo
semanas activas) para ambas versiones, para que la comparación sea justa.

In [7]:
activos_mask = wr["targets"] > 0
test_activos_mask = test_mask & activos_mask

mae_entrenado_con_todas = {}
for nombre, salida in todos_los_resultados.items():
    mae_entrenado_con_todas[nombre] = modeling.metricas(
        y[test_activos_mask], salida["pred"][test_activos_mask]
    )["mae"]

train_activos, val_activos = train_mask & activos_mask, val_mask & activos_mask
resultados_genericos_activos, resumen_genericos_activos = experimentos.entrenar_y_evaluar(
    wr, "receiving_tds", features_arbol, features_ridge, train_activos, val_activos, test_activos_mask
)

comparacion = pd.DataFrame({
    "entrenado_con_todas_evaluado_en_activas": pd.Series(mae_entrenado_con_todas),
    "entrenado_solo_con_activas": resumen_genericos_activos["test_mae"],
}).dropna(subset=["entrenado_solo_con_activas"])
comparacion.round(4)

,entrenado_con_todas_evaluado_en_activas,entrenado_solo_con_activas
Baseline,0.3404,0.3404
HistGradientBoosting,0.3521,0.3605
RandomForest,0.3553,0.3631
Ridge,0.3523,0.3659
XGBoost,0.3515,0.3585


Mismo patrón que en `receptions`/`receiving_yards`: **entrenar con todas las semanas gana en los 5
modelos**, también para `receiving_tds` — las semanas en cero no son ruido a descartar, siguen
aportando contexto real del jugador y su equipo. **Decisión final para los 3 targets: entrenar
siempre con todas las semanas, nunca filtrar por `targets > 0`** — resuelto con evidencia real en
los 2 notebooks de comparación, no asumido por intuición.

## Cierre

`XGBoost (reg:tweedie)` gana por MAE (0.2466 en prueba, contra 0.3071 del mismo XGBoost con
objetivo genérico) y se fija como el modelo de `receiving_tds` para el resto de Fase 5 —
con la salvedad de R² ya documentada arriba, no oculta. Queda también resuelta con evidencia la
pregunta pospuesta de Fase 4 sobre filas sin target: entrenar con todas las semanas, no filtrar,
para los 3 targets.

Notebook anterior: [`13_comparacion_modelos.ipynb`](13_comparacion_modelos.ipynb) · Siguiente:
[`15_estabilidad_temporal.ipynb`](15_estabilidad_temporal.ipynb).